In [1]:
import os
import frontmatter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# 1. Initialize splitter prioritized for Markdown syntax
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=[
        "\n## ",      # Section headings
        "\n### ",     # Sub-section headings
        "\n\n",       # Paragraphs / Blank lines
        "\n",         # Line breaks
        " ",          # Word breaks
        ""
    ]
)

# 2. Configure Ollama embeddings (replace model name with whichever you have pulled)
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",  # Alternatives: llama3, mxbai-embed-large, all-minil
)

def flatten_metadata(meta_dict: dict) -> dict:
    """
    Chroma requires metadata values to be primitive types (str, int, float, bool).
    This converts lists or nested dicts to comma-separated strings.
    """
    clean_meta = {}
    for key, val in meta_dict.items():
        if isinstance(val, list):
            clean_meta[key] = ", ".join(str(item) for item in val)
        elif isinstance(val, (dict, set)):
            clean_meta[key] = str(val)
        elif val is None:
            clean_meta[key] = ""
        else:
            clean_meta[key] = val
    return clean_meta

def load_okf_bundle(base_path: str):
    documents = []

    for root, _, files in os.walk(base_path):
        for file in files:
            if not file.endswith(".md"):
                continue

            file_path = os.path.join(root, file)

            # Extract YAML frontmatter & markdown body cleanly
            post = frontmatter.load(file_path)
            raw_meta = post.metadata
            content = post.content

            # Normalize metadata and add file origin
            clean_meta = flatten_metadata(raw_meta)
            clean_meta["source_path"] = file_path
            clean_meta["category"] = os.path.basename(root)  # 'company', 'policies', or 'products'

            title = clean_meta.get("title", file.replace(".md", ""))
            doc_type = clean_meta.get("type", "general")

            # Split Markdown content
            chunks = text_splitter.split_text(content)

            for i, chunk in enumerate(chunks):
                # Context injection: Prepend document identity to the chunk text
                chunk_text = f"[{title} | Type: {doc_type}]\n{chunk}"

                chunk_meta = clean_meta.copy()
                chunk_meta["chunk_id"] = f"{clean_meta.get('id', file)}_{i}"

                documents.append(
                    Document(
                        page_content=chunk_text,
                        metadata=chunk_meta
                    )
                )

    return documents

# --- Pipeline Execution ---
if __name__ == "__main__":
    bundle_path = "./nexa-ai-okf"

    print("Parsing OKF bundle...")
    docs = load_okf_bundle(bundle_path)
    print(f"Generated {len(docs)} contextual chunks across your folders.")

    # Create & persist Chroma vector store locally
    print("Embedding chunks with Ollama...")
    vector_db = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory="./chroma_okf_db"
    )

    print("Vector database built successfully at ./chroma_okf_db")

    # Quick test query
    retriever = vector_db.as_retriever(search_kwargs={"k": 2})
    results = retriever.invoke("How many days of casual leave do employees get?")

    print("\n--- Test Retrieval ---")
    for r in results:
        print(f"Source: {r.metadata.get('source_path')}")
        print(f"Content:\n{r.page_content}\n")

C:\Users\Ritika Khandelwal\AppData\Local\Temp\ipykernel_31584\3582297392.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings
C:\Users\Ritika Khandelwal\AppData\Local\Temp\ipykernel_31584\3582297392.py:23: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


Parsing OKF bundle...
Generated 23 contextual chunks across your folders.
Embedding chunks with Ollama...
Vector database built successfully at ./chroma_okf_db

--- Test Retrieval ---
Source: ./nexa-ai-okf\policies\leave_policy.md
Content:
[Leave and Time-Off Policy | Type: internal_policy]
# Leave Policy

Structured allocations to sustain employee work-life balance.

## Allocations Per Annum
- **Annual Leave:** 18 working days
- **Sick Leave:** 10 working days
- **Casual Leave:** 6 working days
- **Maternity Leave:** Aligned with prevailing statutory mandates

## Rollover Rules
- Unused annual leave may roll forward up to a maximum cap of **30 calendar days**.

Source: ./nexa-ai-okf\policies\workplace_conduct.md
Content:
[Workplace Conduct & Ethics | Type: compliance_policy]
## Disciplinary Escalation Matrix
Policy breaches result in graduated disciplinary measures:
1. Written formal reprimand
2. Operational suspension
3. Immediate employment termination for severe or willful violatio

In [2]:
for r in results:
    print(f"Source: {r.metadata.get('source_path')}")
    print(f"Related Docs: {r.metadata.get('related_docs')}")
    print(f"Full Metadata: {r.metadata}")

Source: ./nexa-ai-okf\policies\leave_policy.md
Related Docs: policies/hr_policies.md
Full Metadata: {'chunk_id': 'doc_policy_leave_0', 'source_path': './nexa-ai-okf\\policies\\leave_policy.md', 'id': 'doc_policy_leave', 'enforcement': 'internal_hr', 'type': 'internal_policy', 'category': 'policies', 'related_docs': 'policies/hr_policies.md', 'owner': 'company/leadership.md#neha-verma', 'department': 'People Operations', 'title': 'Leave and Time-Off Policy'}
Source: ./nexa-ai-okf\policies\workplace_conduct.md
Related Docs: None
Full Metadata: {'id': 'doc_policy_conduct', 'source_path': './nexa-ai-okf\\policies\\workplace_conduct.md', 'governed_by': 'products/nexasecure.md', 'department': 'Legal & People Operations', 'category': 'policies', 'owner': 'company/leadership.md#neha-verma', 'type': 'compliance_policy', 'title': 'Workplace Conduct & Ethics', 'chunk_id': 'doc_policy_conduct_1'}


In [3]:
# 1. Standard Vector Search (Seed node)
top_hit = retriever.invoke("What are the leave rules?")[0]
print(f"Retrieved Document: {top_hit.metadata.get('title')}")

# 2. Graph Traversal: Follow 'related_docs' stored in metadata
related = top_hit.metadata.get("related_docs")
if related:
    linked_files = [path.strip() for path in related.split(",") if path.strip()]
    print(f"Discovered Dependencies to pull: {linked_files}")
    
    # Query Chroma directly by metadata path to pull the connected docs
    for link in linked_files:
        linked_chunks = vector_db.get(where={"source_path": f"./nexa-ai-okf/{link}"})
        # Now your LLM prompt has both the target policy AND its related context!

Retrieved Document: Leave and Time-Off Policy
Discovered Dependencies to pull: ['policies/hr_policies.md']
